# Retrieving a monophyletic clade

This can be done using the Open Tree identifier for the root of the clade. For example:

In [ ]:
import ete4

t = ete4.Tree('trees/equal_splits_median_tree.tre', parser=1)

for node in t.search_nodes(name='Mammalia_ott244265'):
    mammals = node
    break

mammals.write('examples/my_mammals.tre', parser=1)

# Retrieving a subtree containing only certain species

If you want a tree containing only a subset of species, you can use the "prune" function in ete4. You will need to use taxon identifiers in the same Open Tree format that is used for the labels in the complete tree. These are of the form `Genus_species_ott1234`, where 1234 is the ID of the taxon in the Open Tree Taxonomy.

## I already have a list of Open Tree identifiers

If you already have a list of species labels in this form, retrieving a subtree is straightforward. First, load a complete tree:

In [ ]:
import ete4

t = ete4.Tree('trees/equal_splits_median_tree.tre', parser=1)

Then, prune to your species list:

In [ ]:
t.prune(['Gorilla_gorilla_ott417965',
         'Phyllachora_piptocarphae_ott3705888',
         'Acer_elegantulum_ott1059370',
         'Candidatus_Carsonella_ruddii_ott4790644'], preserve_branch_length=True)

View the tree and write to a Newick file, if you like:

In [ ]:
print(t)
t.write('examples/my_tree.tre', parser=1)

## I need to lookup the Open Tree identifiers

This can be done using the Open Tree Python API. You can install this from your package manager, e.g. `pip install opentree` or `conda install opentree`.

Let's say you have a text file with one species name on each line, like this one in the `examples` folder.

In [ ]:
with open('examples/species_list.txt') as species_file:
    species_list = species_file.read().splitlines()

Now we need to look up the names using the Open Tree taxonomic name resolution service:

In [ ]:
from opentree import OT

response_dict = {}
count = 0
for species_name in species_list:
    print(species_name)
    response = OT.tnrs_match([species_name], context_name="All life", do_approximate_matching=True)
    response_dict[species_name] = response.response_dict
    print("  done", count+1)
    count += 1

The `response_dict` dictionary contains a bunch of information about the taxa it matched. We need to process the results in the dictionary into a list of label names for the complete tree. The following code will work in this simple example, but you may need to modify it depending on the structure of the names you tried to match, and whether you got multiple potential matches for some taxa.

In [ ]:
def construct_ott_name(match):
    name_string = match['taxon']['name']
    name_string = name_string.replace(' ', '_')
    name_string += '_ott'
    name_string += str(match['taxon']['ott_id'])
    return name_string

mynames_to_ottnames = {}

for name in response_dict:
    if len(response_dict[name]["results"][0]["matches"]) == 0:
        print(name, "has no matches")
    else:
        mynames_to_ottnames[name] = construct_ott_name(response_dict[name]["results"][0]["matches"][0])
        print(name, "matched to", mynames_to_ottnames[name])

Now we can load the complete tree, and prune to the species in the list:

In [ ]:
import ete4

t = ete4.Tree('trees/equal_splits_median_tree.tre', parser=1)

In [ ]:
t.prune(list(mynames_to_ottnames.values()), preserve_branch_length=True)